# F02 向量几何、特征与探针


## 这个基础 lab 是为了解决什么

这本 lab 用来消除最常见的小白阻塞项，再进入文章优先主线。


## 做完后你应该带走的技能

- 把 feature 理解成方向、投影和可分性，而不是模糊语义标签。
- 区分相似方向、混叠方向和 probe 可读出的方向。
- 用几何语言解释为什么 probe 能读出某个属性。


## 最后交付这些产物

- 1 张方向、投影和 decision boundary 的示意图。
- 1 段 probe 权重解释。
- 1 份关于几何混叠的 failure note。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/learn-interpretability.git"
REPO_DIR = "learn-interpretability"
REPO_BRANCH = "main"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(candidate)],
            check=True,
        )
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import matplotlib.pyplot as plt
import numpy as np

feature_truth = np.array([1.0, 0.25])
feature_spurious = np.array([0.65, 0.76])
feature_truth = feature_truth / np.linalg.norm(feature_truth)
feature_spurious = feature_spurious / np.linalg.norm(feature_spurious)

rng = np.random.default_rng(5)
positive = rng.normal(loc=[1.2, 0.5], scale=[0.22, 0.18], size=(18, 2))
negative = rng.normal(loc=[-0.9, -0.2], scale=[0.25, 0.2], size=(18, 2))
points = np.vstack([positive, negative])
labels = np.concatenate([np.ones(len(positive)), np.zeros(len(negative))])

centered_labels = labels * 2 - 1
probe = np.linalg.lstsq(points, centered_labels, rcond=None)[0]
probe = probe / np.linalg.norm(probe)

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(positive[:, 0], positive[:, 1], color="#1f5f8b", label="positive")
ax.scatter(negative[:, 0], negative[:, 1], color="#c96a28", label="negative")
for vector, color, label in [
    (feature_truth, "#0f3d5e", "feature_truth"),
    (feature_spurious, "#8b5e34", "feature_spurious"),
    (probe, "#2d6a4f", "probe"),
]:
    ax.arrow(0, 0, vector[0], vector[1], width=0.02, color=color, length_includes_head=True)
    ax.text(vector[0] * 1.08, vector[1] * 1.08, label)
ax.axhline(0, color="0.85")
ax.axvline(0, color="0.85")
ax.set_title("Directions, projections, and a probe")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()

print("cos(feature_truth, probe) =", round(cosine(feature_truth, probe), 3))
print("cos(feature_spurious, probe) =", round(cosine(feature_spurious, probe), 3))
print("Average positive projection onto probe:", round(float((positive @ probe).mean()), 3))
print("Average negative projection onto probe:", round(float((negative @ probe).mean()), 3))


## 小结

一旦把 feature 看成方向，probe 就不再像语义魔法，而更像“几何 + 监督信号”。


## 验收题

- 如果两个特征方向高度相关，probe 的高分为什么仍然可能误导你？
- 什么时候 cosine 相似有用，什么时候它会把结构压扁？
- 把 feature 看成方向之后，你会如何重新解释 M02 里的 monosemanticity？
- 如果你不能在不重开 notebook 的情况下独立答出至少 2 题，就先不要进入文章主线。
